# Trabajo Practico Integrador

**Integrantes:**
1. Hiro Cruz
2. Mauricio Manzano
3. Renzo Sosa
4. Giani Maizón

## Hito 1: Adquisicion y Planteo

### Dataset

**Fuente:** [Student Performance & Behavior Dataset](https://www.kaggle.com/datasets/mahmoudelhemaly/students-grading-dataset)
**Registros:** 5000 estudiantes  
**Variables:** 23 columnas (demograficas, academicas, comportamentales, socioeconomicas)

El dataset contiene registros de rendimiento academico de estudiantes universitarios, incluyendo calificaciones desglosadas por componente (parcial, final, TPs, quizzes, participacion, proyectos), habitos de estudio, nivel de estres, horas de sueno, actividades extracurriculares y contexto socioeconomico familiar.

### Preguntas de Investigacion

1. **Que combinacion de factores comportamentales (asistencia, horas de estudio, estres, sueno) tiene mayor incidencia en la desercion academica (calificacion F), y en que departamentos es mas pronunciada?**

2. **Existe evidencia de que el nivel socioeconomico familiar (ingreso, educacion de padres) impacta el rendimiento academico, o el esfuerzo personal (horas de estudio, participacion) logra compensar esas desventajas?**

3. **Que patrones de discrepancia existen entre el rendimiento en evaluaciones continuas (TPs, quizzes, proyectos) y examenes formales (parcial, final), y como se relacionan con el nivel de estres?**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.conexion import verificar_conexion, inicializar_base
from src.etl import cargar_csv, auditar, limpiar, tratar_outliers, crear_features
from src.visualizaciones import (
    configurar_estilo,
    heatmap_correlacion,
    boxplot_riesgo_calificacion,
    barras_ingreso_esfuerzo,
    scatter_brecha_estres,
    burbuja_estres_sueno,
)

configurar_estilo()

---
## Hito 2: ETL y Calidad de Datos

### Conexion a MySQL y Carga Inicial

In [4]:
RUTA_CSV = 'Students_Performance_Dataset.csv'

df_crudo = cargar_csv(RUTA_CSV)

try:
    engine = inicializar_base()
    df_crudo.to_sql('rendimiento_crudo', engine, if_exists='replace', index=False)
    print(f'Cargados {len(df_crudo)} registros en MySQL (tabla rendimiento_crudo)')
except Exception as e:
    print(f'MySQL no disponible: {e}')
    print('Continuando con datos en memoria')
    engine = None

display(df_crudo.head())

### Auditoria de Datos Crudos

In [5]:
auditar(df_crudo)

### Limpieza de Datos

Proceso de limpieza aplicado:
- Eliminacion de duplicados por `student_id`
- Consolidacion de nombre y apellido en `nombre_completo`; eliminacion de `email`
- Normalizacion de strings: valores categoricos traducidos y estandarizados
- Imputacion de nulos en `educacion_padres` con la moda
- Correccion de precision en columnas numericas (2 decimales)

In [6]:
df_limpio = limpiar(df_crudo)
display(df_limpio.head())

### Deteccion y Tratamiento de Outliers (IQR)

Se aplica el metodo del rango intercuartil (IQR) sobre las variables numericas.  
Los valores fuera del rango [Q1 - 1.5*IQR, Q3 + 1.5*IQR] se capean a los limites correspondientes.

In [7]:
columnas_outliers = [
    'asistencia', 'nota_parcial', 'nota_final', 'promedio_tps',
    'promedio_quizzes', 'puntaje_participacion', 'puntaje_proyectos',
    'puntaje_total', 'horas_estudio_semanal', 'horas_sueno',
]

df_limpio, resumen_outliers = tratar_outliers(df_limpio, columnas_outliers)
display(resumen_outliers)

### Feature Engineering

Variables derivadas creadas para responder a las preguntas de investigacion:
- **promedio_continuo**: Media de TPs, quizzes y proyectos
- **promedio_examenes**: Media de parcial y final
- **brecha_evaluacion**: Diferencia entre evaluacion continua y examenes
- **indice_riesgo**: Indicador compuesto (0-100) basado en asistencia, estres, sueno y horas de estudio
- **categoria_esfuerzo**: Clasificacion del esfuerzo del estudiante (Bajo / Medio / Alto)
- **categoria_bienestar**: Clasificacion del bienestar del estudiante (Bajo / Medio / Alto)

In [8]:
df_final = crear_features(df_limpio)
display(df_final.head())
print(f'\nColumnas finales: {len(df_final.columns)}')
print(df_final.columns.tolist())

### Persistencia de Datos Limpios en MySQL

In [9]:
try:
    if engine is not None:
        df_final.to_sql('rendimiento_limpio', engine, if_exists='replace', index=False)
        print(f'Cargados {len(df_final)} registros en MySQL (tabla rendimiento_limpio)')
    else:
        print('MySQL no disponible. Datos disponibles solo en memoria.')
except Exception as e:
    print(f'Error al persistir datos: {e}')

---
## Hito 3: Visualizacion Dinamica y Analisis

### Estadistica Descriptiva

In [10]:
display(df_final.describe().round(2))

### Visualizacion 1: Matriz de Correlacion

In [11]:
heatmap_correlacion(df_final)

**Analisis:** La matriz revela que las variables con mayor incidencia en el puntaje total son el puntaje de proyectos y la nota del examen final, consistente con sus pesos en la formula de calificacion (30% y 25% respectivamente). Las variables comportamentales (asistencia, horas de estudio, estres, sueno) muestran correlaciones cercanas a cero con el puntaje total, lo que sugiere que el rendimiento academico en este dataset no esta determinado por el esfuerzo observable sino por el desempeno directo en las evaluaciones.

### Visualizacion 2: Indice de Riesgo por Calificacion

In [12]:
boxplot_riesgo_calificacion(df_final)

**Analisis:** La distribucion del indice de riesgo muestra superposicion entre las categorias de calificacion, confirmando que los factores comportamentales por si solos no son predictores confiables del rendimiento. Sin embargo, los estudiantes con calificacion F presentan una mediana de riesgo ligeramente superior, indicando que la combinacion de baja asistencia, alto estres y pocas horas de estudio tiene cierta incidencia en los casos extremos de fracaso academico.

### Visualizacion 3: Puntaje por Ingreso Familiar y Nivel de Esfuerzo

In [13]:
barras_ingreso_esfuerzo(df_final)

**Analisis:** El grafico de barras agrupadas permite evaluar si el esfuerzo personal compensa desventajas socioeconomicas. Se observa que el puntaje total promedio es similar entre los tres niveles de ingreso familiar para una misma categoria de esfuerzo, lo que sugiere que el ingreso familiar no es un factor determinante en el rendimiento academico de este dataset. El nivel de esfuerzo tampoco muestra diferencias marcadas entre categorias, reforzando el hallazgo de que las calificaciones dependen primariamente de las evaluaciones directas.

### Visualizacion 4: Brecha de Evaluacion vs Nivel de Estres

In [14]:
scatter_brecha_estres(df_final)

**Analisis:** El scatter plot muestra la relacion entre la brecha de evaluacion (rendimiento continuo menos examenes) y el nivel de estres, segmentado por departamento. La dispersion es alta, indicando que la brecha esta influenciada por multiples factores mas alla del estres. No se observan diferencias significativas entre departamentos. La linea de regresion permite visualizar la tendencia general de esta relacion en el conjunto de datos.

### Visualizacion 5: Horas de Sueno vs Puntaje Total (Bubble Chart)

In [15]:
burbuja_estres_sueno(df_final)

**Analisis:** El bubble chart presenta tres variables simultaneamente: horas de sueno (eje X), puntaje total (eje Y) y nivel de estres (tamano de burbuja). Las burbujas mas grandes indican mayor nivel de estres. El color diferencia los departamentos. La linea de regresion muestra la tendencia general entre sueno y rendimiento. Se observa alta dispersion sin una correlacion clara entre sueno y puntaje, lo que refuerza el hallazgo de que los factores comportamentales tienen incidencia limitada en el rendimiento academico en este dataset.

---
## Hito 4: Interfaz Grafica y Dashboard Interactivo

En esta etapa se construye una herramienta funcional que permite a un gestor educativo, docente o director explorar los datos sin necesidad de ver codigo. Se utiliza **Streamlit** como framework de desarrollo rapido de interfaces.

El dashboard es **autosuficiente**: aplica el pipeline ETL completo internamente (limpieza, outliers, features) leyendo directamente el CSV crudo. No depende de archivos pre-procesados.

### Especificaciones del Dashboard

El dashboard cumple con los siguientes requisitos:

1. **Interactividad:** Uso de widgets (multiselect, sliders) para filtrar el dataset en tiempo real.

2. **KPIs dinamicos:** Cuatro indicadores principales (Total Estudiantes, Promedio General, Tasa de Aprobacion, % en Riesgo Alto) que se actualizan automaticamente segun los filtros aplicados.

3. **Visualizaciones dinamicas:** Tres graficos que responden a los filtros seleccionados:
    - Distribucion de Calificaciones (barras)
    - Puntaje Total por Departamento (boxplot)
    - Indice de Riesgo por Calificacion (boxplot)

4. **Descarga de datos:** Opcion para exportar los datos filtrados como archivo CSV.

### Ejecucion del Dashboard

```bash
streamlit run src/dashboard.py
```

**Requerimientos:** `streamlit`, `pandas`, `matplotlib`, `seaborn` (ya incluidos en el entorno del proyecto).

**Nota:** Streamlit no se ejecuta de forma interactiva dentro de las celdas del notebook. Se recomienda ejecutarlo en una terminal separada o en un entorno que soporte aplicaciones web.

In [ ]:
# Verificacion: mostrar columnas del df_final para confirmar que el pipeline genero los features
print(f'Dataset final: {len(df_final)} registros, {len(df_final.columns)} columnas')
print('Features derivados:', [c for c in df_final.columns if c in [
    'promedio_continuo', 'promedio_examenes', 'brecha_evaluacion',
    'indice_riesgo', 'categoria_esfuerzo', 'categoria_bienestar'
]])
print('\nPara ejecutar el dashboard: streamlit run src/dashboard.py')

## Hito 5: Informe de Gestion y Propuesta de Mejora

Basados en la evidencia recolectada en los hitos anteriores, se presenta el siguiente informe de gestion.

### 1. Diagnostico Academico

El análisis del dataset de 5000 estudiantes universitarios permite extraer los siguientes hallazgos principales:

A. Baja incidencia de factores comportamentales en el rendimiento. Las variables comportamentales (asistencia, horas de estudio semanal, nivel de estrés y horas de sueño) muestran correlaciones cercanas a cero con el puntaje total. Esto indica que, en este dataset, el desempeño académico no está determinado por hábitos observables. Esta conclusión es robustecida por el solapamiento del indice_riesgo entre las distintas calificaciones y por el bubble chart de estrés y sueño, que muestra alta dispersión sin tendencias claras.

B. Los predictores directos son los componentes de evaluación. La matriz de correlación revela que las variables con mayor asociación al puntaje total son el puntaje de proyectos (30% de peso en la fórmula) y la nota del examen final (25%), lo cual es consistente con su importancia en la composición de la calificación final.

C. El nivel socioeconómico familiar no incide significativamente en el rendimiento. El análisis agrupado por ingreso familiar (Bajo, Medio, Alto) y categoría de esfuerzo (Bajo, Medio, Alto) muestra puntajes promedio similares entre niveles de ingreso para un mismo nivel de esfuerzo. Esto sugiere que, al menos en este dataset, las desventajas socioeconómicas no se traducen automáticamente en peor rendimiento.

D. Alta dispersión en la brecha de evaluación. La diferencia entre el promedio de evaluaciones continuas (TPs, quizzes, proyectos) y el promedio de exámenes (parcial y final) presenta una dispersión elevada sin patrones claros por departamento. Esto indica que el rendimiento en evaluaciones formales es errático y depende de factores no capturados por las variables comportamentales.

E. Limitación: posible origen sintético del dataset. La ausencia casi total de correlaciones entre esfuerzo y rendimiento es atípica en datos educativos reales. Si bien los patrones son internamente consistentes (calificaciones F tienen medianas de riesgo más altas, los componentes pesan según su fórmula), conviene interpretar las conclusiones con cautela.

### 2. Propuestas de Mejora (Justificadas en Datos)

Propuesta A: Sistema de Alerta Temprana basado en Índice de Riesgo

Justificación: Los estudiantes con calificación F presentan una mediana del indice_riesgo superior al resto de las categorías. Si bien existe solapamiento, la tendencia es clara. Implementar un sistema de monitoreo que marque automáticamente a los estudiantes cuyo indice_riesgo supere un umbral crítico (por ejemplo, percentil 90) permitiría intervenir antes de la evaluación final.

Acción: Definir rangos de intervención: indice_riesgo >= 75 (alerta roja, seguimiento individual), 60-74 (alerta naranja, tutoría obligatoria), 50-59 (alerta amarilla, taller de hábitos de estudio).

Propuesta B: Fortalecimiento del Acompañamiento en Evaluaciones Formales

Justificación: La alta dispersión de la brecha_evaluacion (continuo vs. exámenes) sugiere que muchos estudiantes rinden por debajo de su potencial en exámenes formales. El bubble chart de estrés y sueño confirma la dispersión en estas variables comportamentales. Esto puede deberse a ansiedad ante exámenes, falta de práctica en condiciones de presión o deficiencia en técnicas de estudio para evaluación.

Acción: Implementar exámenes simulacro previos a cada evaluación parcial y final, acompañados de talleres de manejo del estrés académico, enfocados en los departamentos con mayor dispersión en la brecha.

Propuesta C: Programa de Acompañamiento Departamental Diferenciado

Justificación: Aunque el análisis general no muestra diferencias departamentales significativas en la relación estrés-brecha, se observan variaciones sutiles en la distribución de puntajes por departamento. Un programa de acompañamiento adaptado a las características propias de cada área (Computación, Ingeniería, Matemáticas, Negocios) permitiría abordar necesidades específicas de cada disciplina.

Acción: Análisis comparativo departamental detallado con chi-cuadrado y ANOVA para determinar si las diferencias son estadísticamente significativas antes de asignar recursos diferenciados.

### 3. Conclusion Final

El análisis realizado sobre los 5000 registros de estudiantes universitarios revela un panorama donde el rendimiento académico depende primariamente del desempeño directo en evaluaciones (proyectos y exámenes finales) y donde los factores comportamentales medidos (asistencia, estudio, estrés, sueño) tienen una influencia limitada en la calificación final. Esto sugiere que las intervenciones más efectivas deberían enfocarse en:

Mejorar el acceso y la calidad de las instancias de evaluación continua, dado que son los componentes con mayor peso relativo.

Desarrollar habilidades de examen en los estudiantes, dado que la brecha entre evaluaciones continuas y formales indica un déficit en el manejo de la presión evaluativa.

Implementar un sistema de alerta temprana basado en el indice_riesgo compuesto, que aunque imperfecto como predictor, proporciona una señal temprana accesible para intervenciones preventivas.

La limitación más importante es que los datos presentan características de dataset sintético, por lo que estas propuestas deberían validarse con datos reales antes de su implementación a escala institucional.